# Get Indy crash data

Indiana crash data lives in the ARIES database (https://www.ariesportal.com/) which provides data for free for gov agencies but not for the general public or even research groups

But one can still get it for Indianapolis.

### From Indianapolis MPO Data portal's dashboard for serious crashes:

- Link to dashboard: https://data-indympo.hub.arcgis.com/apps/83d778fc586a4a43aba848a494b1cda3/explore
- While they don't provide download links, the dashboard relies on ArcGIS FeatureLayers (can inspect in the browser's DevTools -> Network tab)
- The relevant FeatureLayer is called "2018-2025 ARIES Cleaned" (*it DOES contain 2026 data*): https://services5.arcgis.com/qVN2o0aio8BMbwcJ/arcgis/rest/services/2018_2025_ARIES_Cleaned__Update/FeatureServer?f=json
- Download 2000 records at a time (use &resultOffset=2000, 4000 etc): https://services5.arcgis.com/qVN2o0aio8BMbwcJ/arcgis/rest/services/2018_2025_ARIES_Cleaned__Update/FeatureServer/0/query?where=1=1&outFields=*&f=geojson

### Alternatively, from SafeStreetsIndy:

https://safestreetsindy.org/

- 2026: https://data.safestreetsindy.org/safestreetsindy/incidents-2026.json?h=d5ec15a2a6e818caf703cc58d3b90863 
- 2025: https://data.safestreetsindy.org/safestreetsindy/incidents-2025.json?h=4c6b8ee961bc23d979bbd14e9f6708a4
- 2024: https://data.safestreetsindy.org/safestreetsindy/incidents-2024.json?h=a6c01299817de303bd521b7721a37aaa
- 2023: https://data.safestreetsindy.org/safestreetsindy/incidents-2023.json?h=cec71986eaa5149a03bc5e112e7f2529
- 2022: https://data.safestreetsindy.org/safestreetsindy/incidents-2022.json?h=7ce37e9b61143daee5aed6645212e978 

In [1]:
import pandas as pd
import json
import requests

In [2]:
feature_layer_url = 'https://services5.arcgis.com/qVN2o0aio8BMbwcJ/arcgis/rest/services/\
2018_2025_ARIES_Cleaned__Update/FeatureServer/0/query'

params = {
    'where': '1=1',
    'outFields': '*',
    'f': 'geojson'
}

In [3]:
dfs = []

for offset in range(0, 11000, 2000):
    
    params['resultOffset'] = offset
    
    dfs.append(
        pd.DataFrame(
            json.loads(
                requests.get(feature_layer_url, params).text
            )['features']
        )
    )

In [4]:
crashes = pd.concat(dfs).assign(
    lat=lambda df_: df_.geometry.str['coordinates'].str[1],
    lon=lambda df_: df_.geometry.str['coordinates'].str[0],
    id=lambda df_: df_.properties.str['Master_Record_Number'],
    county=lambda df_: df_.properties.str['County'],
    city=lambda df_: df_.properties.str['City'],
    date=lambda df_: df_.properties.str['Date'],
    year=lambda df_: df_.properties.str['Year'],
    crash_type=lambda df_: df_.properties.str['Crash_Type'],
    cross_street=lambda df_: df_.properties.str['Cross_Streeet'],
    manner=lambda df_: df_.properties.str['Manner_Of_Collision'],
    primary_factor=lambda df_: df_.properties.str['Primary_Factor'],
    ssi_fatal=lambda df_: df_.properties.str['SSI_Fatal'],
    light=lambda df_: df_.properties.str['Light_Condition'],
    weather=lambda df_: df_.properties.str['Weather_Condition'],
    surface=lambda df_: df_.properties.str['Surface_Condition']
).filter([
    'id', 'year', 'date', 'county', 'city', 'cross_street', 'crash_type',
    'manner', 'primary_factor', 'ssi_fatal', 'light', 'weather', 'surface',
    'lat', 'lon'
])

In [5]:
crashes.to_csv('crashes.csv', index=False)

### Let's look at values

In [6]:
crashes.ssi_fatal.value_counts()

ssi_fatal
SSI      7758
Fatal    1634
Name: count, dtype: int64

In [7]:
crashes.crash_type.value_counts()

crash_type
Vehicle       8081
Pedestrian    1061
Pedalcycle     250
Name: count, dtype: int64

In [8]:
crashes.weather.value_counts()

weather
CLEAR                       6869
CLOUDY                      1440
RAIN                         820
SNOW                         127
SLEET/HAIL/FREEZING RAIN      45
FOG/SMOKE/SMOG                45
BLOWING SAND/SOIL/SNOW        31
SEVERE CROSS WIND             11
Name: count, dtype: int64

In [9]:
crashes.light.value_counts()

light
DAYLIGHT              5217
DARK (LIGHTED)        2121
DARK (NOT LIGHTED)    1645
DAWN/DUSK              398
UNKNOWN                  7
Name: count, dtype: int64

In [10]:
crashes.primary_factor.value_counts()

primary_factor
FAILURE TO YIELD RIGHT OF WAY            1909
OTHER (DRIVER) - EXPLAIN IN NARRATIVE     879
DISREGARD SIGNAL/REG SIGN                 739
UNSAFE SPEED                              722
PEDESTRIAN ACTION                         622
                                         ... 
I-65 NB NEAR 104.6MM                        1
W SOUTH ST & S MISSOURI ST                  1
N RILEY HWY                                 1
S SR 39 & LINCOLN ST                        1
I-65 NB                                     1
Name: count, Length: 138, dtype: int64

In [11]:
crashes.manner.value_counts()

manner
RAN OFF ROAD                                2010
RIGHT ANGLE                                 1839
REAR END                                    1286
OTHER - EXPLAIN IN NARRATIVE                1119
HEAD ON BETWEEN TWO MOTOR VEHICLES           955
LEFT TURN                                    710
SAME DIRECTION SIDESWIPE                     421
NON-COLLISION                                319
COLLISION WITH OBJECT IN ROAD                196
OPPOSITE DIRECTION SIDESWIPE                 129
RIGHT TURN                                    92
LEFT/RIGHT TURN                               84
BACKING CRASH                                 47
COLLISION WITH DEER                           28
FAILURE TO YIELD RIGHT OF WAY                 20
COLLISION WITH ANIMAL OTHER                   18
FAILURE TO MAINTAIN LANE                      11
OTHER (DRIVER) - EXPLAIN IN NARRATIVE         10
REAR TO REAR                                   9
UNSAFE SPEED                                   7
UNSAFE LANE M